In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/mariaworkman/fashion/fashion-neutrality")
sys.path.insert(0, str(PROJECT_ROOT))

print("using project root:", PROJECT_ROOT)

using project root: /Users/mariaworkman/fashion/fashion-neutrality


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/mariaworkman/VSCode25/fashion-neutrality")

IN_CSV = PROJECT_ROOT / "/Users/mariaworkman/fashion/fashion-neutrality/data/samples/sample_2000_2023_100peryear.csv"
IMG_DIR = PROJECT_ROOT / "/Users/mariaworkman/fashion/fashion-neutrality/data/images/sample_2000_2023_100peryear"

df = pd.read_csv(IN_CSV)
df["local_path"] = df["filename"].apply(lambda f: str(IMG_DIR / f))
df["download_ok"] = df["local_path"].apply(lambda p: Path(p).exists())
df = df[df["download_ok"]].copy()

print("images ready:", len(df))
df.head()

images ready: 2399


,key,designer,season,year,category,city,section,tags,aesthetic,url,filename,local_path,download_ok
0,655677,Rebecca Danenberg,Spring,2000,Ready-to-Wear,NaN,Collection,['Clothing' 'Apparel' 'Fashion' 'Person' 'Runw...,4.89,https://assets.vogue.com/photos/55c6514f08298d...,0655677_Rebecca Danenberg - Spring 2000 Ready-...,/Users/mariaworkman/fashion/fashion-neutrality...,True
1,560590,Miu Miu,Spring,2000,Ready-to-Wear,NaN,Collection,['Long Sleeve' 'Clothing' 'Apparel' 'Female' '...,4.24,https://assets.vogue.com/photos/55c6514e08298d...,0560590_Miu Miu - Spring 2000 Ready-to-Wear [C...,/Users/mariaworkman/fashion/fashion-neutrality...,True
2,444360,Halston,Fall,2000,Ready-to-Wear,NaN,Collection,['Long Sleeve' 'Clothing' 'Apparel' 'Person' '...,4.07,https://assets.vogue.com/photos/55c6516e08298d...,0444360_Halston - Fall 2000 Ready-to-Wear [Col...,/Users/mariaworkman/fashion/fashion-neutrality...,True
3,980151,Helmut Lang,Fall,2000,Ready-to-Wear,NaN,Collection,['Long Sleeve' 'Clothing' 'Apparel' 'Fashion' ...,4.46,https://assets.vogue.com/photos/5a3aba11216e3e...,0980151_Helmut Lang - Fall 2000 Ready-to-Wear ...,/Users/mariaworkman/fashion/fashion-neutrality...,True
4,68211,Loewe,Spring,2000,Ready-to-Wear,NaN,Collection,['Evening Dress' 'Clothing' 'Apparel' 'Fashion...,3.81,https://assets.vogue.com/photos/55c6514e08298d...,0068211_Loewe - Spring 2000 Ready-to-Wear [Col...,/Users/mariaworkman/fashion/fashion-neutrality...,True


In [3]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from PIL import Image
from sklearn.cluster import MiniBatchKMeans

from rembg import remove  

from src.image_utils import sample_pixels_from_mask
from src.color_utils import kmeans_palette_lab
from src.metrics import neutrality_metrics

# params
N_CLUSTERS = 8
SAMPLE_N = 20000          # pixels sampled per image (tradeoff: speed vs stability)
RANDOM_STATE = 42

OUT_CSV_REMBG = "../data/results/metrics_2010_2023_100peryear_rembg.csv"
os.makedirs(Path(OUT_CSV_REMBG).parent, exist_ok=True)

/Users/mariaworkman/fashion/fashion-neutrality/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np
import cv2
import io
from PIL import Image
from rembg import remove, new_session

session = new_session("u2net")

def rembg_mask_binarized(bgr_img, session):
    """
    Returns HxW uint8 mask (0 background, 255 foreground)
    Works whether rembg.remove returns ndarray or PNG bytes.
    """
    rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)

    out = remove(rgb, session=session)

    # Case 1: rembg returns ndarray
    if isinstance(out, np.ndarray):
        arr = out
        # could be RGB or RGBA
        if arr.ndim == 3 and arr.shape[2] == 4:
            alpha = arr[:, :, 3]
            return (alpha > 0).astype(np.uint8) * 255
        else:
            # no alpha channel; treat non-black as foreground
            return ((arr.sum(axis=2) > 0).astype(np.uint8)) * 255

    # Case 2: rembg returns bytes (PNG)
    if isinstance(out, (bytes, bytearray)):
        im = Image.open(io.BytesIO(out)).convert("RGBA")
        alpha = np.array(im)[:, :, 3]
        return (alpha > 0).astype(np.uint8) * 255

    raise TypeError(f"Unexpected rembg output type: {type(out)}")

Background removal ensures chroma measurements reflect garment color rather than runway lighting/backgrounds

In [5]:
bgr = cv2.imread(df.iloc[0]["local_path"])
mask = rembg_mask_binarized(bgr, session)
print(mask.shape, mask.dtype, np.unique(mask)[:10])

(360, 240) uint8 [  0 255]


In [ ]:
N_CLUSTERS = 8
SAMPLE_N = 20000
RANDOM_STATE = 42
TAU = 20.0

results = []

for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
    local_path = row["local_path"]

    out = {
        "key": row.get("key"),
        "designer": row.get("designer"),
        "season": row.get("season"),
        "year": row.get("year"),
        "category": row.get("category"),
        "city": row.get("city"),
        "section": row.get("section"),
        "tags": row.get("tags"),
        "aesthetic": row.get("aesthetic"),
        "url": row.get("url"),
        "filename": row.get("filename"),
        "local_path": local_path,
        "ok": False,
        "error": None,
    }

    try:
        bgr = cv2.imread(local_path, cv2.IMREAD_COLOR)
        if bgr is None:
            raise ValueError("cv2.imread returned None")

        mask = rembg_mask_binarized(bgr, session)

        pixels_bgr = sample_pixels_from_mask(
            bgr, mask, n=SAMPLE_N, seed=RANDOM_STATE + i
        )

        if pixels_bgr is None:
            pixels_bgr = sample_pixels_full(
                bgr, n=SAMPLE_N, seed=RANDOM_STATE + i
            )

        centers_lab, weights = kmeans_palette_lab(
            pixels_bgr,
            k=N_CLUSTERS,
            seed=RANDOM_STATE
        )

        mean_chroma, neutral_share = neutrality_metrics(
            centers_lab=centers_lab,
            weights=weights,
            tau=TAU
        )

        out["mean_chroma"] = mean_chroma
        out["neutral_share"] = neutral_share
        out["ok"] = True

    except Exception as e:
        out["error"] = str(e)

    results.append(out)

metrics_df = pd.DataFrame(results)

OUT_CSV = PROJECT_ROOT / "data/results/metrics_full_run.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(OUT_CSV, index=False)

In [ ]:
print(OUT_CSV.resolve())